In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-white")
plt.rcParams.update({
    "font.family": "Helvetica",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "legend.frameon": False,
    "legend.fontsize": 7,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

In [ ]:
from tbh.paths import REPO_ROOT_PATH, DATA_FOLDER
analysis_path = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "52039665_fixed_matrix" / "task_1"


In [ ]:
import tbh.plotting as pl
import tbh.runner_tools as rt
import pandas as pd

In [ ]:
intervention_scenarios = [sc.sc_id for sc in rt.SCENARIOS]
all_scenarios = ['baseline'] + intervention_scenarios
unc_dfs = {
    sc: pd.read_parquet(analysis_path / f"uncertainty_df_{sc}.parquet") for sc in all_scenarios
}
diff_outputs_dfs = {
    sc: pd.read_parquet(analysis_path / f"diff_quantiles_df_ref_baseline_{sc}.parquet") for sc in intervention_scenarios
}

In [ ]:
import yaml

with open(analysis_path / "details.yaml" , "r") as f:
    docs = list(yaml.safe_load_all(f))

model_config = docs[1]
analysis_config = docs[2]

In [ ]:
from tbh.model import get_tb_model
from estival.model import BayesianCompartmentalModel

params, priors, tv_params = rt.get_parameters_and_priors()

model = get_tb_model(model_config, tv_params)
bcm = BayesianCompartmentalModel(model, params, priors, rt.targets)

In [ ]:

from importlib import reload
reload(pl)

In [ ]:
from math import ceil
import matplotlib.lines as mlines

colour = "#B22222"

def make_figure_1(uncertainty_df, bcm):

    n_col = 3
    selected_outputs = ['pearl_pos_per100k', 'cxr_pos_per100k', 'perc_prev_subclinical', 'perc_prev_infectious', 'notifications']
    n_panels = len(selected_outputs) + 1
    n_row = ceil(n_panels / n_col)

    fig, axes = plt.subplots(n_row, n_col, figsize=(5 * n_col, 3.6 * n_row))
    axes = axes.flatten()  # Flatten to simplify indexing

    for i, output in enumerate(selected_outputs):
        ax = axes[i]
        out_name = output if output not in pl.title_lookup else pl.title_lookup[output]
        x_min = 1990 if output == "notifications" else 2010
        pl.plot_model_fit_with_uncertainty(ax, uncertainty_df, output, bcm, x_lim=(x_min, 2025), colour=colour)
        # ax.set_title(out_name)
        if i == 0:
            ax.legend()


   # Add a panel with TBI prevalence by agegroup
    ax=axes[len(selected_outputs)]
    agegroups=["3_9","10","15+","18+"]
    model_median=[]
    model_low=[]
    model_high=[]
    observed=[]
    x_tick_labels=[]
    for i_age,age in enumerate(agegroups):
        output_name=f"tst_posXage_{age}_perc"
        year=bcm.targets[output_name].data.index[0]
        q=uncertainty_df[output_name].loc[year]
        obs=bcm.targets[output_name].data.iloc[0]
        model_median.append(q["0.5"])
        model_low.append(q["0.025"])
        model_high.append(q["0.975"])
        observed.append(obs)
        suffix=f" y.o.\n({year})"
        if age=="3_9":
            x_tick_labels.append("3–9"+suffix)
        elif age=="15+":
            x_tick_labels.append("15+"+suffix)
        else:
            x_tick_labels.append(f"{age}"+suffix)
    x=range(len(agegroups))
    ax.errorbar(
        [i - 0.06 for i in x],
        model_median,
        yerr=[
            [m-l for m,l in zip(model_median,model_low)],
            [h-m for h,m in zip(model_high,model_median)]
        ],
        fmt='D',
        color=colour,
        ecolor=colour,
        markersize=4,
        elinewidth=2.,
        capsize=0,
        label="Model (median, 95% CI)"
    )
    ax.scatter(
        [i + 0.06 for i in x],
        observed,
        color="black",
        s=15,
        zorder=5,
        label="Observed"
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(x_tick_labels)
    ax.set_ylabel(pl.title_lookup["tst_pos_perc"])
    # ax.set_title("TBI prevalence by age group")
    ax.grid(alpha=0.3)

    model_handle=mlines.Line2D([],[],color=colour,marker='D',markersize=4,linestyle='-',label='Model (median, 95% CI)')
    obs_handle=mlines.Line2D([],[],color='black',marker='o',linestyle='None',markersize=4,label='Observed')
    ax.legend(handles=[obs_handle, model_handle],frameon=False,loc='best')


    fig.tight_layout()

    return fig


make_figure_1(unc_dfs['baseline'], bcm)

plt.savefig("manuscript_figs/figure_1.png", dpi=300)
plt.savefig("manuscript_figs/figure_1.pdf")


In [ ]:
from matplotlib import patches as mpatches


def add_scenario_annotations_(ax, x_sep_positions, labels):

    # Get top of y-axis for label placement
    ymax = ax.get_ylim()[1]

    prev = 0
    for i, label in enumerate(labels):
        # Vertical line
        vline_pos = x_sep_positions[i]
        if i < len(x_sep_positions):
            ax.axvline(x=vline_pos, linestyle='--', linewidth=0.5, color='black')

        # # Text slightly above the plot
        ax.text(
            prev + (vline_pos - prev) / 2,
            ymax * 1.01,       # a bit above the top
            label,
            ha='center',
            va='bottom',
            rotation=0,
            clip_on=False
        )

        prev = vline_pos


def plot_diff_outputs_(axis, diff_quantiles_dfs, output_name, scenarios):

    box_width = .4
    med_color = 'white'
    box_color= 'green'
    y_max_abs = 0.

    sc_names = ["55%", "65%", "75%", "85%", "95%"] * 3


    for i, sc in enumerate(scenarios):

        diff_output_df = diff_quantiles_dfs[sc]
        data = diff_output_df[output_name] 
        
        if output_name.endswith("_relative"):  # use %
            data = data * 100.

        # use %. And use "-" so positive nbs indicate positive effect of closures
        x = 1 + i
        # median
        axis.hlines(y=data.loc[0.5], xmin=x - box_width / 2. , xmax= x + box_width / 2., lw=.5, color=med_color, zorder=3)    
        
        # IQR
        q_75 = data.loc[0.75]
        q_25 = data.loc[0.25]
        rect = mpatches.Rectangle(xy=(x - box_width / 2., q_25), width=box_width, height=q_75 - q_25, zorder=2, facecolor=box_color)
        axis.add_patch(rect)

        # 95% CI
        q_025 = data.loc[0.025]
        q_975 = data.loc[0.975]
        axis.vlines(x=x, ymin=q_025 , ymax=q_975, lw=.8, color=box_color, zorder=1)

        y_max_abs = max(abs(q_975), y_max_abs)
        y_max_abs = max(abs(q_025), y_max_abs)
 
    y_label = output_name if output_name not in pl.title_lookup else pl.title_lookup[output_name]  
    axis.set_ylabel(y_label)
   
    axis.set_xlabel("Screening coverage")


    x_labels = sc_names
    axis.set_xticks(ticks=range(1, len(scenarios) + 1), labels=x_labels) #, fontsize=15)

    axis.set_xlim((0.5, len(scenarios) + 0.5))
    axis.set_ylim(0., 1.05 * y_max_abs)

In [ ]:
int_scenarios = [f"scenario_{i}" for i in list(range(1, 11)) + [16,17,18,19,20]]

In [ ]:
for output in ["TB_averted_relative", "deaths_averted_relative"]:
    fig, ax = plt.subplots(1, 1, figsize=(0.5*len(int_scenarios), 3.))
    plot_diff_outputs_(ax, diff_outputs_dfs, output, int_scenarios)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

    add_scenario_annotations_(ax, [5.5, 10.5, 15.5] , ["PEARL", "Drop Xpert", "Drop Xpert & TST"])

    ax.grid(axis='y', linestyle='-', linewidth=0.7, alpha=0.4)

    plt.savefig(f"manuscript_figs/{output}_diffs.png", dpi=300)
    plt.savefig(f"manuscript_figs/{output}_diffs.pdf")
